# Strategy 3 grid search — which forward flattener is long vol *and* paid to hold?

The companion to `strat3_strikeless_vol_backtest.ipynb`, which runs one
config through the production engine. This notebook runs the whole grid and
ranks it.

**The question, in the user's own words:**

> "strat 3 is about finding structures that allow us to get **LONG VOL and
> EARN THETA** via the forward curve structures."

So this is not a Sharpe hunt. Every one of the fifteen pairs is long gamma —
that is a theorem about `dM = M_long - M_short`, not an empirical finding, and
section 4 checks it holds on every pair on every day. What separates them is
**carry**: the roll a flattener pays for the privilege of being convex. The
ranking table therefore carries `mean_carry_1y_bp`, `pct_days_carry_positive`
and `mean_gamma_usd_bp2` next to the Sharpe, and **a Sharpe-optimal cell that
is bleeding carry is flagged rather than crowned**.

## What is searched

| axis | values | why |
|---|---|---|
| pair | the Citi 15-pair grid | `10y10y/{15y15y,20y5y,20y10y,20y15y,25y5y,25y10y}`, `15y5y/{20y5y,20y10y,20y15y,25y5y,25y10y}`, `15y10y/{25y5y,25y10y}`, `20y5y/{25y5y,25y10y}` |
| `hedge_threshold_bp` | 10, 15, 20, 25, 30, 40 | Citi: "the Sharpe ratio ... doesn't change significantly if the threshold is chosen in the **15-30bp** range, but declines with a smaller or larger threshold" — a prediction this grid can falsify |
| `resize_mode` | neutral, always_decrease | the PM's two exits: "delta hedge it to 0 risk (always decrease) or keep it constant" |
| `beta` | 1.0, 1.025 | Citi's Oct-2019 reweight of the live 15y5y/20y10y trade |
| `roll_months` | 12, 24 | Citi rolled "every year"; 24 tests whether the ageing helps or hurts |
| entry rule | always-on, z-gated, BE/rv-gated, both, carry-gated | Doc A §8.1's own screen, made mechanical |

## The reference point

Citi Figure 4, sample 12/31/2013-5/7/2019, $100K DV01, 25bp hedging, net of
their costs: **10y5y/15y15y 0.05, 10y10y/15y15y 0.13, 10y10y/20y10y 0.16,
10y10y/20y15y 0.24, 10y10y/25y10y 0.35, 15y5y/20y10y 0.18, 15y5y/20y15y 0.25,
20y5y/25y10y 0.30.** Different sample (ours is 2019-2026), same size, same
hedging rule, same cost schedule. Their number appears as a column on every
row it exists for, so the comparison is never left to memory.

In [1]:
import os

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

import datetime
import itertools
import json
import pathlib
import sys
import time

import numpy as np
import pandas as pd

_REPO = pathlib.Path(__file__).resolve().parents[3] if "__file__" in dir() else pathlib.Path.cwd().parents[2]
sys.path.insert(0, str(_REPO))

from BT.data_handler import TimeGrid
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

import RVUtils.ConvexityRV.strat3_strikeless_vol as S3
from RVUtils.ConvexityRV.curve_ops import payoff_profile

DATA = _REPO / "notebooks" / "data" / "convexity_rv"
CURVE = "USD-SOFR-1D"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

C:\Users\chris\clee\ARBS-cvx\Query\IRSwaps\IRSwapStructure.py:12: LicenceNotice: 
Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence
  from rateslib.scheduling import get_imm, next_imm


## 1. The config — the grid axes

In [2]:
GRID = {
    # --- WHAT ------------------------------------------------------------
    "pairs": list(S3.PAIRS_15),

    # --- THE HEDGE -------------------------------------------------------
    "thresholds": [10.0, 15.0, 20.0, 25.0, 30.0, 40.0],   # bp move in the LONGER rate
    "resize_modes": ["neutral", "always_decrease"],
    "betas": [1.0, 1.025],
    "roll_months": [12, 24],

    # --- THE ENTRY GATE (lag-1, scaling an always-on aged ledger) --------
    "entry_rules": [
        {"entry_rule": "always"},
        {"entry_rule": "z", "z_window": "1y", "z_min": 1.0},
        {"entry_rule": "z", "z_window": "3y", "z_min": 1.0},
        {"entry_rule": "be_ratio", "be_ratio_max": 0.80},
        {"entry_rule": "be_ratio", "be_ratio_max": 0.50},
        {"entry_rule": "z_and_be", "z_window": "3y", "z_min": 0.5, "be_ratio_max": 0.80},
        {"entry_rule": "carry", "carry_min_bp": 0.0},      # long gamma AND paid: the thesis
    ],

    # --- SIZING AND COSTS ------------------------------------------------
    "package_dv01_usd": 100_000.0,
    "cost_multipliers": [0.0, 1.0, 2.0],   # gross | Citi Fig-9 | capacity stress

    # --- SAMPLE ----------------------------------------------------------
    "start": datetime.date(2019, 1, 1),
    "end": datetime.date(2026, 8, 14),
}
PRIMARY_COST = 1.0            # the multiplier every headline number is quoted at
SPAN_YEARS = (pd.Timestamp(GRID["end"]) - pd.Timestamp(GRID["start"])).days / 365.25
print(json.dumps({k: (str(v) if isinstance(v, datetime.date) else v)
                  for k, v in GRID.items() if k != "pairs"}, indent=1))
print(f"pairs {len(GRID['pairs'])}, span_years {SPAN_YEARS:.2f}")

{
 "thresholds": [
  10.0,
  15.0,
  20.0,
  25.0,
  30.0,
  40.0
 ],
 "resize_modes": [
  "neutral",
  "always_decrease"
 ],
 "betas": [
  1.0,
  1.025
 ],
 "roll_months": [
  12,
  24
 ],
 "entry_rules": [
  {
   "entry_rule": "always"
  },
  {
   "entry_rule": "z",
   "z_window": "1y",
   "z_min": 1.0
  },
  {
   "entry_rule": "z",
   "z_window": "3y",
   "z_min": 1.0
  },
  {
   "entry_rule": "be_ratio",
   "be_ratio_max": 0.8
  },
  {
   "entry_rule": "be_ratio",
   "be_ratio_max": 0.5
  },
  {
   "entry_rule": "z_and_be",
   "z_window": "3y",
   "z_min": 0.5,
   "be_ratio_max": 0.8
  },
  {
   "entry_rule": "carry",
   "carry_min_bp": 0.0
  }
 ],
 "package_dv01_usd": 100000.0,
 "cost_multipliers": [
  0.0,
  1.0,
  2.0
 ],
 "start": "2019-01-01",
 "end": "2026-08-14"
}
pairs 15, span_years 7.62


## 2. The planted-answer sign test, live

Re-measured on every execution. `bpv < 0` on a CURVE package must be the
flattener, and the flattener must be the convex leg; a regression in
`resolve_pricable` would invert the entire grid without raising.

In [3]:
def _run_sign_probe(bpv: float) -> float:
    dates = [datetime.date(2022, 9, 12), datetime.date(2022, 9, 13),
             datetime.date(2022, 9, 14), datetime.date(2022, 9, 15)]
    q = IRSwapQuery(structure=IRSwapStructure.OUTRIGHT, value=IRSwapValue.NPV,
                    tenor="5Y", curve=CURVE, structure_kwargs={"bpv": bpv}, tags=("probe",))
    strat = QueryStrategy(name=f"sign_{bpv:+.0f}", triggers=[
        DateTrigger(DateTriggerRequirements(dates=[dates[0]]),
                    actions=[AddQueryAction(query=q, meta={"tags": ["probe"]})]),
        DateTrigger(DateTriggerRequirements(dates=[dates[-1]]),
                    actions=[UnwindPositionsAction(match_tag="probe", fee=0.0)])])
    bt = QueryDrivenBacktest(time_grid=TimeGrid([pd.Timestamp(d) for d in dates]),
                             strategy=strat, mdp=IRSwapsMDP(source="CITIVELO_EXCEL"),
                             show_progress=False)
    bt.run()
    return float(pd.Series(bt.mtm_history).iloc[-1])


plus, minus = _run_sign_probe(+100_000.0), _run_sign_probe(-100_000.0)
print(f"+bpv {plus:+,.0f}   -bpv {minus:+,.0f}")
assert plus > 0, "payer must gain in the 2022-09 selloff"
assert abs(plus + minus) < 1e-6 * abs(plus), "buy/sell must mirror -- seam regressed?"

_mdp = IRSwapsMDP(source="CITIVELO_EXCEL")
_p22 = _mdp.get_pricer({"curve_name": CURVE, "timestamp": datetime.date(2022, 9, 13),
                        "offline": True})
_q = IRSwapQuery(structure=IRSwapStructure.CURVE, value=IRSwapValue.NPV, curve=CURVE,
                 structure_kwargs={"front_tenor": "10Yx10Y", "back_tenor": "20Yx10Y",
                                   "bpv": -100_000.0})
_pk, _w = _q.resolve_package(pricer_or_curve=_p22)
_pk = [_p22.resolve_pricable(x, rw) for x, rw in zip(_pk, _w)]
_prof = payoff_profile(_p22, _pk, [-250, -200, -150, -100, -50, -25, 0, 25, 50, 100, 150, 200, 250],
                       horizon_date=None) / 100_000.0
print(f"10Yx10Y/20Yx10Y flattener payoff, bp: {np.round(_prof, 1).tolist()}")
assert (np.diff(_prof, 2) > 0).all(), "bpv<0 CURVE is not long gamma -- sign inverted"
assert np.allclose(np.round(_prof, 1),
                   [104.9, 60.0, 29.9, 11.6, 2.3, 0.4, 0.0, 0.9, 2.9, 9.5, 18.7, 29.7, 41.7],
                   atol=0.05), "the committed payoff regression table has moved"
print("SIGN + CONVEXITY + REGRESSION TABLE PASS")

C:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning: invalid value encountered in divide
  r = (v[:-1] / v[1:] - 1) * 100 / dcfs_obs[len(populated) :]


+bpv +2,303,346   -bpv -2,303,346


10Yx10Y/20Yx10Y flattener payoff, bp: [104.9, 60.0, 29.9, 11.6, 2.3, 0.4, 0.0, 0.9, 2.9, 9.5, 18.7, 29.7, 41.7]
SIGN + CONVEXITY + REGRESSION TABLE PASS


## 3. The precomputed panels

`scripts/strat3_build_panels.py` produced both. The screen is constant
maturity; the ledgers hold aged packages. Ledgers are stored at **zero cost**
with the traded RISK recorded separately, so every cost multiplier below is a
re-charge of the same path rather than a re-run of it.

In [4]:
SCREEN = pd.read_parquet(DATA / "strat3_screen.parquet")
LEDGERS_RAW = pd.read_parquet(DATA / "strat3_ledgers.parquet")

LEDGERS = {}
for key, grp in LEDGERS_RAW.groupby(level=["pair", "roll_months", "threshold_bp",
                                           "beta", "resize_mode"]):
    pair, rm, th, beta, mode = key
    # plain python types: the parquet round trip gives numpy scalars, and the
    # lookup key ``run_grid`` builds comes off a dataclass holding python ones
    LEDGERS.setdefault(tuple(pair.split("/")), {})[(int(rm), float(th), float(beta), str(mode))] = (
        grp.droplevel(["pair", "roll_months", "threshold_bp", "beta", "resize_mode"]).sort_index())

SCREENS = {p: SCREEN.xs(p, level="pair").sort_index()
           for p in SCREEN.index.get_level_values("pair").unique()}

_v = LEDGERS_RAW.reset_index()[["roll_months", "threshold_bp", "beta", "resize_mode"]].drop_duplicates()
print(f"screen  {SCREEN.shape}  {SCREEN.index.get_level_values('date').min().date()} .. "
      f"{SCREEN.index.get_level_values('date').max().date()}")
print(f"ledgers {LEDGERS_RAW.shape}  {len(LEDGERS)} pairs x {len(_v)} hedging variants")
_any = next(iter(next(iter(LEDGERS.values())).values()))
print(f"ledger dates {_any.index.min().date()} .. {_any.index.max().date()} ({len(_any)} days)")
assert len(LEDGERS) == 15, f"expected 15 pairs, got {sorted(LEDGERS)}"
assert len(_v) == len(GRID["thresholds"]) * len(GRID["resize_modes"]) * len(GRID["betas"]) \
    * len(GRID["roll_months"]), "the cached variant set does not match the grid"

screen  (50955, 17)  2013-01-02 .. 2026-08-07
ledgers (1366560, 15)  15 pairs x 48 hedging variants
ledger dates 2019-01-02 .. 2026-08-07 (1898 days)


## 4. Known-answer checks

Three, before any ranking is read.

**(a) Citi Figure 7**, close of 2019-05-08 — the external known answer, eight
published levels and carries.

**(b)/(c) DV01 neutrality and convexity, on every pair on every day** of the
3,397-day screen. Not a spot check: `package_dv01 ~ 0` and `gamma > 0` are
structural claims and the panel can test them 50,955 times.

**(d) The sign probe** — section 2.

In [5]:
_P = _mdp.get_pricer({"curve_name": CURVE, "timestamp": datetime.date(2019, 5, 8),
                      "offline": True})
FIG7 = S3.screen_frame(_P, list(S3.CITI_FIG7_SCREEN), asof=datetime.date(2019, 5, 8),
                       include_query_carry=True).set_index("pair")
_pub = pd.DataFrame([{"pair": f"{a}/{b}", "citi_level": v[0], "citi_carry": v[3]}
                     for (a, b), v in S3.CITI_FIG7_SCREEN.items()]).set_index("pair")
TIE = FIG7.join(_pub)
TIE["d_level"] = TIE.level_bp - TIE.citi_level
TIE["d_carry"] = TIE.carry_1y_bp - TIE.citi_carry
display(TIE[["level_bp", "citi_level", "d_level", "carry_1y_bp", "citi_carry", "d_carry",
             "carry_query_bp", "gamma_ratio", "be_daily_analytic", "be_daily_exact"]].round(3))

_scr_all = SCREEN
checks = [
    ("(a) Fig 7 levels within 1.5bp", TIE.d_level.abs().max() < 1.5, TIE.d_level.abs().max()),
    ("(a) Fig 7 carry within 1.0bp", TIE.d_carry.abs().max() < 1.0, TIE.d_carry.abs().max()),
    ("(b) package PV01 ~ 0 on 2019-05-08",
     FIG7.package_dv01_usd.abs().max() < 1e-6 * 100_000.0, FIG7.package_dv01_usd.abs().max()),
    ("(c) gamma > 0 on every pair, every day",
     bool((_scr_all.gamma_usd_bp2 > 0).all()), float(_scr_all.gamma_usd_bp2.min())),
    ("(c) repriced gamma == dM/1e4 within 10%, every day",
     bool(_scr_all.gamma_ratio.between(0.90, 1.10).all()),
     f"{_scr_all.gamma_ratio.min():.3f}..{_scr_all.gamma_ratio.max():.3f}"),
    ("BE(analytic) == BE(repriced) within 5% where carry < 0",
     bool((_scr_all.loc[_scr_all.carry_1y_bp < 0, "be_daily_exact"]
           / _scr_all.loc[_scr_all.carry_1y_bp < 0, "be_daily_analytic"]).between(0.95, 1.05).all()),
     float((_scr_all.loc[_scr_all.carry_1y_bp < 0, "be_daily_exact"]
            / _scr_all.loc[_scr_all.carry_1y_bp < 0, "be_daily_analytic"]).max())),
    ("BE is exactly 0 wherever carry >= 0",
     bool((_scr_all.loc[_scr_all.carry_1y_bp >= 0, "be_daily_analytic"] == 0).all()),
     float(_scr_all.loc[_scr_all.carry_1y_bp >= 0, "be_daily_analytic"].max())),
]
display(pd.DataFrame(checks, columns=["check", "ok", "value"]))
assert all(ok for _, ok, _ in checks), "known-answer checks FAILED"
print(f"KNOWN-ANSWER PASS ({len(_scr_all):,} pair-days checked for (b)/(c))")

,level_bp,citi_level,d_level,carry_1y_bp,citi_carry,d_carry,carry_query_bp,gamma_ratio,be_daily_analytic,be_daily_exact
pair,,,,,,,,,,
10Yx5Y/15Yx15Y,-11.405,-10.1,-1.305,-2.426,-2.91,0.484,0.170,0.975,3.103,3.141
10Yx10Y/15Yx15Y,-9.657,-8.8,-0.857,-1.292,-1.64,0.348,-0.107,0.983,2.614,2.636
10Yx10Y/20Yx10Y,-14.246,-13.2,-1.046,-1.294,-1.75,0.456,-0.272,1.017,2.266,2.247
10Yx10Y/20Yx15Y,-17.454,-16.7,-0.754,-1.442,-1.88,0.438,-0.079,0.998,2.139,2.142
10Yx10Y/25Yx10Y,-20.757,-20.1,-0.657,-1.444,-1.83,0.386,-0.108,1.018,1.955,1.937
15Yx5Y/20Yx10Y,-12.264,-11.3,-0.964,-0.009,-0.31,0.301,-0.554,0.997,0.224,0.224
15Yx5Y/20Yx15Y,-15.472,-15.2,-0.272,-0.157,-0.44,0.283,-0.361,0.978,0.789,0.798
20Yx5Y/25Yx10Y,-8.854,-9.1,0.246,-0.005,0.13,-0.135,0.421,0.998,0.165,0.165


,check,ok,value
0,(a) Fig 7 levels within 1.5bp,True,1.305253
1,(a) Fig 7 carry within 1.0bp,True,0.484052
2,(b) package PV01 ~ 0 on 2019-05-08,True,0.0
3,"(c) gamma > 0 on every pair, every day",True,101.375712
4,"(c) repriced gamma == dM/1e4 within 10%, every...",True,0.947..1.058
5,BE(analytic) == BE(repriced) within 5% where c...,True,1.027239
6,BE is exactly 0 wherever carry >= 0,True,0.0


KNOWN-ANSWER PASS (50,955 pair-days checked for (b)/(c))


## 5. The ex-ante screen, by pair

Before any P&L: which pairs are structurally paid to be long gamma? This is
the table the brief's objective is really about, and it splits the universe
into two families that no amount of hedge tuning will merge.

In [6]:
_w = SCREEN[(SCREEN.index.get_level_values("date") >= pd.Timestamp(GRID["start"]))
            & (SCREEN.index.get_level_values("date") <= pd.Timestamp(GRID["end"]))]
EXANTE = _w.groupby("pair").agg(
    mean_level_bp=("level_bp", "mean"),
    mean_carry_1y_bp=("carry_1y_bp", "mean"),
    pct_days_carry_pos=("carry_1y_bp", lambda s: 100 * (s >= 0).mean()),
    mean_gamma_usd_bp2=("gamma_usd_bp2", "mean"),
    mean_be_daily_bp=("be_daily_analytic", "mean"),
    mean_rlzd_vol_bp=("rlzd_vol_bp", "mean"),
    mean_be_over_rv=("be_over_rv", "mean"),
).round(3)
EXANTE["dM_years"] = [S3.delta_m_years(*p.split("/")) for p in EXANTE.index]
EXANTE["citi_fig4_sharpe"] = [S3.CITI_FIG4_SHARPE.get(tuple(p.split("/")), np.nan)
                              for p in EXANTE.index]
display(EXANTE.sort_values("mean_carry_1y_bp", ascending=False))

,mean_level_bp,mean_carry_1y_bp,pct_days_carry_pos,mean_gamma_usd_bp2,mean_be_daily_bp,mean_rlzd_vol_bp,mean_be_over_rv,dM_years,citi_fig4_sharpe
pair,,,,,,,,,
15Yx5Y/20Yx5Y,-33.616,0.087,46.207,101.715,1.491,5.010,0.309,5.0,NaN
15Yx5Y/20Yx10Y,-44.280,-0.064,40.464,149.520,1.456,4.900,0.316,7.5,0.18
15Yx5Y/25Yx5Y,-56.625,-0.263,39.937,203.459,1.435,4.772,0.325,10.0,NaN
15Yx10Y/25Yx5Y,-41.107,-0.288,43.941,156.015,1.304,4.772,0.304,7.5,NaN
20Yx5Y/25Yx5Y,-23.009,-0.350,58.535,101.744,1.316,4.772,0.310,5.0,NaN
15Yx5Y/20Yx15Y,-55.854,-0.380,33.720,195.749,1.552,4.740,0.345,10.0,0.25
15Yx5Y/25Yx10Y,-69.518,-0.682,28.609,251.631,1.632,4.570,0.379,12.5,NaN
15Yx10Y/25Yx10Y,-54.000,-0.707,24.183,204.187,1.685,4.570,0.395,10.0,NaN
20Yx5Y/25Yx10Y,-35.902,-0.769,27.608,149.916,1.721,4.570,0.412,7.5,0.30


## 6. The grid

Every cell is scored off the cached zero-cost ledgers — no repricing happens
here, which is what makes a grid this size affordable. Sharpe is Citi's own
Figure-4 convention: `mean(daily $) / std(daily $) x sqrt(252)`, computed on
every day in the sample including flat ones, so a gated book and an always-on
one are on the same scale.

In [7]:
CELLS = S3.grid_cells(GRID["pairs"], GRID["thresholds"], GRID["resize_modes"],
                      GRID["betas"], GRID["roll_months"], GRID["entry_rules"])
print(f"{len(CELLS):,} cells x {len(GRID['cost_multipliers'])} cost multipliers")

t0 = time.time()
frames = []
for mult in GRID["cost_multipliers"]:
    g = S3.run_grid(LEDGERS, SCREENS, CELLS, package_dv01_usd=GRID["package_dv01_usd"],
                    cost_multiplier=mult, window=(GRID["start"], GRID["end"]))
    g["cost_multiplier"] = mult
    frames.append(g)
RESULTS = pd.concat(frames, ignore_index=True)
print(f"scored {len(RESULTS):,} rows in {time.time()-t0:.0f}s")
assert len(RESULTS) == len(CELLS) * len(GRID["cost_multipliers"]), "cells went missing"
RESULTS.to_csv(DATA / "strat3_grid_results.csv", index=False)
print(f"-> {DATA / 'strat3_grid_results.csv'}")

R1 = RESULTS[RESULTS.cost_multiplier == PRIMARY_COST].copy()
COLS = ["pair", "threshold_bp", "resize_mode", "beta", "roll_months", "entry_rule",
        "be_ratio_max", "sharpe", "sharpe_ex_mtm", "total_net_usd", "net_bp",
        "carry_bp", "harvest_bp", "mtm_bp", "net_ex_mtm_bp",
        "mean_carry_1y_bp", "pct_days_carry_positive",
        "mean_gamma_usd_bp2", "mean_be_over_rv", "occupancy", "n_hedges", "skew",
        "hit_rate", "max_dd_usd", "citi_fig4_sharpe"]

5,040 cells x 3 cost multipliers


scored 15,120 rows in 68s


-> C:\Users\chris\clee\ARBS-cvx\notebooks\data\convexity_rv\strat3_grid_results.csv


### 6.1 Top 25 by annualised Sharpe — with the carry column attached

`mean_carry_1y_bp` is the ex-ante roll of the structure and
`pct_days_carry_positive` is how often it was actually paid. `carry_bp` is
the realised roll bucket of the book itself. **Read those before the Sharpe.**

In [8]:
TOP = R1.sort_values("sharpe", ascending=False).head(25)[COLS]
TOP.insert(0, "flag", np.where(TOP.mean_carry_1y_bp >= 0, "PAID",
                               np.where(TOP.mean_carry_1y_bp > -1.0, "cheap", "BLEEDS")))
display(TOP.round(3).reset_index(drop=True))

_best = R1.loc[R1.sharpe.idxmax()]
print(f"\nSharpe-optimal cell: {_best.pair} @{_best.threshold_bp:g}bp "
      f"{_best.resize_mode} beta={_best.beta} roll={_best.roll_months}m "
      f"entry={_best.entry_rule}")
print(f"  Sharpe {_best.sharpe:.3f}, net ${_best.total_net_usd:,.0f}, "
      f"mean 1y carry {_best.mean_carry_1y_bp:+.2f}bp "
      f"({_best.pct_days_carry_positive*100:.0f}% of days >= 0)")
if _best.mean_carry_1y_bp < 0:
    print("  ** FLAG: this cell is SHORT theta. It is long gamma (every cell is), "
          "but it pays to be, which is not the structure the brief asks for.")

,flag,pair,threshold_bp,resize_mode,beta,roll_months,entry_rule,be_ratio_max,sharpe,sharpe_ex_mtm,total_net_usd,net_bp,carry_bp,harvest_bp,mtm_bp,net_ex_mtm_bp,mean_carry_1y_bp,pct_days_carry_positive,mean_gamma_usd_bp2,mean_be_over_rv,occupancy,n_hedges,skew,hit_rate,max_dd_usd,citi_fig4_sharpe
0,BLEEDS,10Yx10Y/25Yx10Y,40.0,neutral,1.025,12,be_ratio,0.8,0.765,-0.066,1.038686e+07,103.869,-11.427,29.970,112.882,-9.013,-4.170,0.020,306.123,0.720,0.662,18.0,0.264,0.518,-3466182.368,0.35
1,BLEEDS,10Yx10Y/25Yx10Y,40.0,neutral,1.000,12,be_ratio,0.8,0.741,-0.069,1.032242e+07,103.224,-11.015,29.239,112.533,-9.309,-4.170,0.020,306.123,0.720,0.662,18.0,0.283,0.511,-3528694.860,0.35
2,BLEEDS,10Yx10Y/25Yx10Y,30.0,neutral,1.025,12,be_ratio,0.8,0.724,-0.114,9.748616e+06,97.486,-11.429,23.778,112.882,-15.396,-4.170,0.020,306.123,0.720,0.662,28.0,0.194,0.530,-3443991.321,0.35
3,BLEEDS,10Yx10Y/25Yx10Y,15.0,neutral,1.025,12,be_ratio,0.8,0.722,-0.121,9.618336e+06,96.183,-11.483,23.373,112.882,-16.699,-4.170,0.020,306.123,0.720,0.662,89.0,0.286,0.529,-3540715.326,0.35
4,cheap,15Yx5Y/25Yx10Y,40.0,neutral,1.000,12,be_ratio,0.8,0.713,-0.123,1.165979e+07,116.598,-1.130,25.344,130.622,-14.024,-0.682,0.286,251.631,0.379,0.861,20.0,-0.142,0.525,-2565623.185,NaN
5,cheap,15Yx5Y/25Yx10Y,40.0,neutral,1.025,12,be_ratio,0.8,0.712,-0.121,1.140799e+07,114.080,-1.825,25.978,128.186,-14.106,-0.682,0.286,251.631,0.379,0.861,20.0,-0.177,0.530,-2467513.781,NaN
6,cheap,15Yx5Y/25Yx10Y,30.0,neutral,1.025,24,be_ratio,0.8,0.710,-0.164,1.086389e+07,108.639,-5.172,8.900,142.263,-33.624,-0.682,0.286,251.631,0.379,0.861,35.0,-0.314,0.537,-2325458.529,NaN
7,cheap,15Yx5Y/25Yx10Y,30.0,neutral,1.000,24,be_ratio,0.8,0.708,-0.166,1.105739e+07,110.574,-4.512,8.683,143.727,-33.153,-0.682,0.286,251.631,0.379,0.861,35.0,-0.268,0.519,-2424119.393,NaN
8,cheap,15Yx5Y/25Yx10Y,15.0,neutral,1.025,24,be_ratio,0.8,0.707,-0.170,1.070720e+07,107.072,-5.289,8.315,142.263,-35.191,-0.682,0.286,251.631,0.379,0.861,109.0,-0.347,0.534,-2357334.622,NaN
9,BLEEDS,10Yx10Y/25Yx10Y,10.0,neutral,1.025,12,be_ratio,0.8,0.704,-0.137,9.398362e+06,93.984,-11.468,21.754,112.882,-18.898,-4.170,0.020,306.123,0.720,0.662,162.0,0.244,0.533,-3579323.782,0.35



Sharpe-optimal cell: 10Yx10Y/25Yx10Y @40bp neutral beta=1.025 roll=12m entry=be_ratio
  Sharpe 0.765, net $10,386,864, mean 1y carry -4.17bp (2% of days >= 0)
  ** FLAG: this cell is SHORT theta. It is long gamma (every cell is), but it pays to be, which is not the structure the brief asks for.


### 6.2 The same table ranked the way the brief asks

Sharpe **subject to** the structure being paid to hold: filtered to cells
whose mean ex-ante 1y carry is non-negative. If this table is empty, no
structure in the Citi grid earned theta over 2019-2026 and that is itself the
answer.

In [9]:
PAID = R1[R1.mean_carry_1y_bp >= 0].sort_values("sharpe", ascending=False)
print(f"{len(PAID):,} of {len(R1):,} cells are long gamma AND non-negative carry "
      f"({100*len(PAID)/len(R1):.1f}%)")
display(PAID.head(20)[COLS].round(3).reset_index(drop=True))

NEAR = R1[R1.mean_carry_1y_bp >= -0.5].sort_values("sharpe", ascending=False)
print(f"\nrelaxed to carry >= -0.5bp/yr: {len(NEAR):,} cells")
display(NEAR.head(15)[COLS].round(3).reset_index(drop=True))

336 of 5,040 cells are long gamma AND non-negative carry (6.7%)


,pair,threshold_bp,resize_mode,beta,roll_months,entry_rule,be_ratio_max,sharpe,sharpe_ex_mtm,total_net_usd,net_bp,carry_bp,harvest_bp,mtm_bp,net_ex_mtm_bp,mean_carry_1y_bp,pct_days_carry_positive,mean_gamma_usd_bp2,mean_be_over_rv,occupancy,n_hedges,skew,hit_rate,max_dd_usd,citi_fig4_sharpe
0,15Yx5Y/20Yx5Y,15.0,neutral,1.000,12,always,0.8,0.335,0.186,4362912.755,43.629,0.150,13.284,35.063,8.567,0.087,0.462,101.715,0.309,1.0,148.0,-0.456,0.527,-3249535.599,NaN
1,15Yx5Y/20Yx5Y,15.0,always_decrease,1.000,12,always,0.8,0.321,0.158,4334704.060,43.347,1.222,11.013,35.063,8.285,0.087,0.462,101.715,0.309,1.0,24.0,-0.398,0.524,-3590961.645,NaN
2,15Yx5Y/20Yx5Y,10.0,always_decrease,1.000,12,always,0.8,0.320,0.158,4333038.386,43.330,1.252,10.969,35.063,8.268,0.087,0.462,101.715,0.309,1.0,34.0,-0.396,0.524,-3540029.236,NaN
3,15Yx5Y/20Yx5Y,10.0,neutral,1.000,12,always,0.8,0.315,0.131,4105441.672,41.054,0.158,10.917,35.063,5.992,0.087,0.462,101.715,0.309,1.0,253.0,-0.458,0.529,-3287785.427,NaN
4,15Yx5Y/20Yx5Y,25.0,always_decrease,1.000,12,always,0.8,0.312,0.132,4195382.582,41.954,1.028,9.798,35.063,6.891,0.087,0.462,101.715,0.309,1.0,11.0,-0.422,0.524,-3660473.775,NaN
5,15Yx5Y/20Yx5Y,30.0,always_decrease,1.000,12,always,0.8,0.310,0.126,4164278.880,41.643,1.008,9.501,35.063,6.580,0.087,0.462,101.715,0.309,1.0,9.0,-0.436,0.523,-3658033.323,NaN
6,15Yx5Y/20Yx5Y,20.0,always_decrease,1.000,12,always,0.8,0.308,0.122,4145564.426,41.456,1.085,9.247,35.063,6.393,0.087,0.462,101.715,0.309,1.0,16.0,-0.418,0.526,-3616020.958,NaN
7,15Yx5Y/20Yx5Y,20.0,neutral,1.000,12,always,0.8,0.304,0.100,3966797.665,39.668,0.172,8.935,35.063,4.605,0.087,0.462,101.715,0.309,1.0,79.0,-0.446,0.524,-3320012.863,NaN
8,15Yx5Y/20Yx5Y,15.0,neutral,1.025,12,always,0.8,0.304,0.171,3934004.634,39.340,-0.637,13.616,31.255,8.085,0.087,0.462,101.715,0.309,1.0,148.0,-0.527,0.535,-3437286.109,NaN
9,15Yx5Y/20Yx5Y,30.0,neutral,1.000,12,always,0.8,0.300,0.088,3904616.710,39.046,0.151,8.114,35.063,3.984,0.087,0.462,101.715,0.309,1.0,37.0,-0.452,0.528,-3373977.449,NaN



relaxed to carry >= -0.5bp/yr: 2,016 cells


,pair,threshold_bp,resize_mode,beta,roll_months,entry_rule,be_ratio_max,sharpe,sharpe_ex_mtm,total_net_usd,net_bp,carry_bp,harvest_bp,mtm_bp,net_ex_mtm_bp,mean_carry_1y_bp,pct_days_carry_positive,mean_gamma_usd_bp2,mean_be_over_rv,occupancy,n_hedges,skew,hit_rate,max_dd_usd,citi_fig4_sharpe
0,15Yx10Y/25Yx5Y,10.0,neutral,1.000,12,be_ratio,0.8,0.699,0.087,7706679.277,77.067,2.137,19.118,70.823,6.243,-0.288,0.439,156.015,0.304,0.826,205.0,-0.609,0.525,-1697610.420,NaN
1,15Yx10Y/25Yx5Y,10.0,neutral,1.000,24,be_ratio,0.8,0.695,-0.011,7498109.536,74.981,2.713,9.728,76.418,-1.437,-0.288,0.439,156.015,0.304,0.826,207.0,-0.688,0.523,-1697610.420,NaN
2,15Yx10Y/25Yx5Y,30.0,neutral,1.000,12,be_ratio,0.8,0.694,0.080,7656586.593,76.566,2.157,17.653,70.823,5.742,-0.288,0.439,156.015,0.304,0.826,34.0,-0.569,0.524,-1666806.120,NaN
3,15Yx10Y/25Yx5Y,15.0,neutral,1.000,24,be_ratio,0.8,0.688,-0.019,7399860.985,73.999,2.663,8.366,76.418,-2.419,-0.288,0.439,156.015,0.304,0.826,108.0,-0.671,0.523,-1688730.354,NaN
4,15Yx10Y/25Yx5Y,30.0,neutral,1.000,24,be_ratio,0.8,0.687,-0.020,7391015.076,73.910,2.727,7.657,76.418,-2.508,-0.288,0.439,156.015,0.304,0.826,34.0,-0.667,0.524,-1730619.266,NaN
5,15Yx10Y/25Yx5Y,10.0,neutral,1.025,24,be_ratio,0.8,0.687,-0.013,7321404.365,73.214,2.204,9.971,74.959,-1.745,-0.288,0.439,156.015,0.304,0.826,207.0,-0.744,0.530,-1734715.778,NaN
6,15Yx10Y/25Yx5Y,15.0,neutral,1.000,12,be_ratio,0.8,0.686,0.063,7532952.298,75.330,2.085,17.009,70.823,4.506,-0.288,0.439,156.015,0.304,0.826,108.0,-0.603,0.524,-1688730.354,NaN
7,15Yx10Y/25Yx5Y,10.0,neutral,1.025,12,be_ratio,0.8,0.685,0.084,7460320.856,74.603,1.575,19.595,68.484,6.120,-0.288,0.439,156.015,0.304,0.826,205.0,-0.675,0.533,-1688526.453,NaN
8,15Yx10Y/25Yx5Y,30.0,neutral,1.025,12,be_ratio,0.8,0.680,0.077,7408975.855,74.090,1.596,18.095,68.484,5.606,-0.288,0.439,156.015,0.304,0.826,34.0,-0.620,0.532,-1639579.270,NaN
9,15Yx10Y/25Yx5Y,15.0,neutral,1.025,24,be_ratio,0.8,0.679,-0.021,7220699.600,72.207,2.153,8.575,74.959,-2.752,-0.288,0.439,156.015,0.304,0.826,108.0,-0.725,0.527,-1700461.956,NaN


### 6.3 By pair, always-on, at Citi's own 25bp — the like-for-like comparison

One row per pair, at exactly Citi's specification (25bp, DV01-neutral, annual
roll, always on, $100K DV01, their cost schedule). Their Figure-4 Sharpe sits
in the last column. Different sample — theirs is 12/2013-5/2019, ours is
1/2019-8/2026 — so this is a robustness read, not a reproduction.

In [10]:
BASE = R1[(R1.threshold_bp == 25.0) & (R1.resize_mode == "neutral")
          & (R1.beta == 1.0) & (R1.roll_months == 12)
          & (R1.entry_rule == "always")].set_index("pair")
LIKE = BASE[["sharpe", "sharpe_ex_mtm", "total_net_usd", "net_bp", "carry_bp",
             "harvest_bp", "mtm_bp", "net_ex_mtm_bp",
             "mean_carry_1y_bp", "pct_days_carry_positive", "mean_gamma_usd_bp2",
             "mean_be_over_rv", "n_hedges", "skew", "hit_rate", "max_dd_usd",
             "citi_fig4_sharpe"]].sort_values("sharpe", ascending=False)
display(LIKE.round(3))

_both = LIKE.dropna(subset=["citi_fig4_sharpe"])
print(f"\n{len(_both)} pairs overlap Citi's Figure 4.")
print(f"  ours  mean {_both.sharpe.mean():+.3f}, range {_both.sharpe.min():+.3f}..{_both.sharpe.max():+.3f}")
print(f"  Citi  mean {_both.citi_fig4_sharpe.mean():+.3f}, "
      f"range {_both.citi_fig4_sharpe.min():+.3f}..{_both.citi_fig4_sharpe.max():+.3f}")
if len(_both) > 2:
    print(f"  cross-sectional rank corr (Spearman): "
          f"{_both[['sharpe','citi_fig4_sharpe']].corr(method='spearman').iloc[0,1]:+.3f}")

,sharpe,sharpe_ex_mtm,total_net_usd,net_bp,carry_bp,harvest_bp,mtm_bp,net_ex_mtm_bp,mean_carry_1y_bp,pct_days_carry_positive,mean_gamma_usd_bp2,mean_be_over_rv,n_hedges,skew,hit_rate,max_dd_usd,citi_fig4_sharpe
pair,,,,,,,,,,,,,,,,,
15Yx10Y/25Yx10Y,0.578,0.005,7869444.054,78.694,-6.181,11.320,78.205,0.490,-0.707,0.242,204.187,0.395,38.0,-0.027,0.521,-2496182.040,NaN
15Yx5Y/25Yx10Y,0.546,0.024,9788897.859,97.889,-6.238,13.745,95.234,2.655,-0.682,0.286,251.631,0.379,38.0,-0.212,0.524,-3977528.284,NaN
20Yx5Y/25Yx10Y,0.536,-0.027,5827754.598,58.278,-6.259,8.789,60.171,-1.894,-0.769,0.276,149.916,0.412,38.0,-0.012,0.531,-1401372.544,0.30
15Yx5Y/20Yx15Y,0.505,0.053,7779971.937,77.800,-3.626,13.114,73.053,4.746,-0.380,0.337,195.749,0.345,46.0,-0.328,0.523,-3723288.016,0.25
15Yx10Y/25Yx5Y,0.470,0.052,5937357.174,59.374,-2.592,10.810,55.656,3.718,-0.288,0.439,156.015,0.304,42.0,-0.324,0.518,-2572699.634,NaN
20Yx5Y/25Yx5Y,0.464,0.007,3793409.179,37.934,-2.662,7.221,37.622,0.312,-0.350,0.585,101.744,0.310,42.0,-0.398,0.519,-1388298.389,NaN
15Yx5Y/25Yx5Y,0.447,0.073,7944514.200,79.445,-2.661,14.145,72.685,6.761,-0.263,0.399,203.459,0.325,42.0,-0.346,0.522,-3916480.059,NaN
15Yx5Y/20Yx10Y,0.404,0.108,6018557.909,60.186,-1.028,11.718,52.905,7.281,-0.064,0.405,149.520,0.316,49.0,-0.385,0.520,-3527730.921,0.18
10Yx10Y/25Yx10Y,0.404,-0.159,6891071.836,68.911,-32.115,15.857,90.255,-21.344,-4.170,0.020,306.123,0.720,38.0,0.107,0.506,-6740920.665,0.35



7 pairs overlap Citi's Figure 4.
  ours  mean +0.363, range +0.112..+0.536
  Citi  mean +0.230, range +0.130..+0.350
  cross-sectional rank corr (Spearman): +0.714


### 6.4 The gamma line on its own — with the directional windfall removed

**This is the most important table in the notebook, and it is the one that
most changes the reading of section 6.3.**

Over 2019-2026 the long-end forward curve inverted enormously: the mean
10y10y/20y10y level went from about −14bp in May 2019 to about −56bp in
August 2026. A flattener held through that made a fortune **for a reason that
has nothing to do with convexity** — it was long the move. That P&L lands in
the `mtm` bucket, and on most pairs it is larger than every other bucket put
together.

So the columns to read are:

* `harvest_bp` — the P&L of the resizes alone. This is the strategy.
* `carry_bp` — the roll the position paid to exist. This is the theta bill.
* `net_ex_mtm_bp` = `harvest + carry − costs` — **the delta-hedged vol trade
  with the direction stripped out.** Positive means the gamma paid for its own
  theta and its own transaction costs. That is exactly the claim the note
  makes and the brief asks for.
* `mtm_bp` — where the curve went. Not a skill.

`sharpe_ex_mtm` is the annualised Sharpe of that same direction-free series.

In [11]:
GAMMA_VIEW = BASE[["harvest_bp", "carry_bp", "mtm_bp", "net_ex_mtm_bp", "net_bp",
                   "sharpe_ex_mtm", "sharpe", "n_hedges", "mean_gamma_usd_bp2",
                   "mean_carry_1y_bp"]].copy()
GAMMA_VIEW["harvest_over_carry"] = (GAMMA_VIEW.harvest_bp / GAMMA_VIEW.carry_bp.abs()).replace(
    [np.inf, -np.inf], np.nan)
GAMMA_VIEW["harvest_per_hedge_usd"] = (BASE.harvest_usd / BASE.n_hedges).round(0)
GAMMA_VIEW["mtm_share_of_net"] = (BASE.mtm_bp / BASE.net_bp).round(3)
display(GAMMA_VIEW.sort_values("net_ex_mtm_bp", ascending=False).round(3))

_pos = GAMMA_VIEW[GAMMA_VIEW.net_ex_mtm_bp > 0]
print(f"\n{len(_pos)}/{len(GAMMA_VIEW)} pairs are profitable with the curve move removed "
      f"(harvest + carry - costs > 0), at Citi's own 25bp specification.")
print(f"mean mtm share of net across the 15 pairs: "
      f"{GAMMA_VIEW.mtm_share_of_net.mean():.2f}")

,harvest_bp,carry_bp,mtm_bp,net_ex_mtm_bp,net_bp,sharpe_ex_mtm,sharpe,n_hedges,mean_gamma_usd_bp2,mean_carry_1y_bp,harvest_over_carry,harvest_per_hedge_usd,mtm_share_of_net
pair,,,,,,,,,,,,,
15Yx5Y/20Yx10Y,11.718,-1.028,52.905,7.281,60.186,0.108,0.404,49.0,149.520,-0.064,11.394,23915.0,0.879
15Yx5Y/25Yx5Y,14.145,-2.661,72.685,6.761,79.445,0.073,0.447,42.0,203.459,-0.263,5.315,33678.0,0.915
15Yx5Y/20Yx15Y,13.114,-3.626,73.053,4.746,77.800,0.053,0.505,46.0,195.749,-0.380,3.617,28508.0,0.939
15Yx10Y/25Yx5Y,10.810,-2.592,55.656,3.718,59.374,0.052,0.470,42.0,156.015,-0.288,4.170,25737.0,0.937
15Yx5Y/20Yx5Y,6.952,0.159,35.063,2.794,37.857,0.061,0.290,49.0,101.715,0.087,43.662,14188.0,0.926
15Yx5Y/25Yx10Y,13.745,-6.238,95.234,2.655,97.889,0.024,0.546,38.0,251.631,-0.682,2.203,36171.0,0.973
15Yx10Y/25Yx10Y,11.320,-6.181,78.205,0.490,78.694,0.005,0.578,38.0,204.187,-0.707,1.831,29790.0,0.994
20Yx5Y/25Yx5Y,7.221,-2.662,37.622,0.312,37.934,0.007,0.464,42.0,101.744,-0.350,2.713,17193.0,0.992
20Yx5Y/25Yx10Y,8.789,-6.259,60.171,-1.894,58.278,-0.027,0.536,38.0,149.916,-0.769,1.404,23129.0,1.032



8/15 pairs are profitable with the curve move removed (harvest + carry - costs > 0), at Citi's own 25bp specification.
mean mtm share of net across the 15 pairs: 1.29


### 6.5 Ranked the way the brief asks, twice over

`sharpe_ex_mtm` ranks the **vol trade**; `mean_carry_1y_bp >= 0` restricts to
structures that are **paid to hold**. A cell that clears both is the thing
the objective actually names: long gamma, earning theta, with the direction
taken out.

In [12]:
BOTH = R1[(R1.mean_carry_1y_bp >= 0) & (R1.net_ex_mtm_bp > 0)].sort_values(
    "sharpe_ex_mtm", ascending=False)
print(f"{len(BOTH):,} of {len(R1):,} cells are long gamma, non-negative carry, AND "
      f"profitable ex-direction")
display(BOTH.head(20)[COLS].round(3).reset_index(drop=True))

VOLONLY = R1.sort_values("sharpe_ex_mtm", ascending=False)
print("\ntop 15 by direction-free Sharpe, carry unconstrained:")
display(VOLONLY.head(15)[COLS].round(3).reset_index(drop=True))

24 of 5,040 cells are long gamma, non-negative carry, AND profitable ex-direction


,pair,threshold_bp,resize_mode,beta,roll_months,entry_rule,be_ratio_max,sharpe,sharpe_ex_mtm,total_net_usd,net_bp,carry_bp,harvest_bp,mtm_bp,net_ex_mtm_bp,mean_carry_1y_bp,pct_days_carry_positive,mean_gamma_usd_bp2,mean_be_over_rv,occupancy,n_hedges,skew,hit_rate,max_dd_usd,citi_fig4_sharpe
0,15Yx5Y/20Yx5Y,15.0,neutral,1.000,12,always,0.8,0.335,0.186,4362912.755,43.629,0.150,13.284,35.063,8.567,0.087,0.462,101.715,0.309,1.0,148.0,-0.456,0.527,-3249535.599,NaN
1,15Yx5Y/20Yx5Y,15.0,neutral,1.025,12,always,0.8,0.304,0.171,3934004.634,39.340,-0.637,13.616,31.255,8.085,0.087,0.462,101.715,0.309,1.0,148.0,-0.527,0.535,-3437286.109,NaN
2,15Yx5Y/20Yx5Y,10.0,always_decrease,1.000,12,always,0.8,0.320,0.158,4333038.386,43.330,1.252,10.969,35.063,8.268,0.087,0.462,101.715,0.309,1.0,34.0,-0.396,0.524,-3540029.236,NaN
3,15Yx5Y/20Yx5Y,15.0,always_decrease,1.000,12,always,0.8,0.321,0.158,4334704.060,43.347,1.222,11.013,35.063,8.285,0.087,0.462,101.715,0.309,1.0,24.0,-0.398,0.524,-3590961.645,NaN
4,15Yx5Y/20Yx5Y,10.0,always_decrease,1.025,12,always,0.8,0.295,0.145,3903383.406,39.034,0.493,11.243,31.255,7.779,0.087,0.462,101.715,0.309,1.0,34.0,-0.474,0.522,-3696102.164,NaN
5,15Yx5Y/20Yx5Y,15.0,always_decrease,1.025,12,always,0.8,0.295,0.145,3905090.721,39.051,0.462,11.289,31.255,7.796,0.087,0.462,101.715,0.309,1.0,24.0,-0.477,0.522,-3748307.883,NaN
6,15Yx5Y/20Yx5Y,25.0,always_decrease,1.000,12,always,0.8,0.312,0.132,4195382.582,41.954,1.028,9.798,35.063,6.891,0.087,0.462,101.715,0.309,1.0,11.0,-0.422,0.524,-3660473.775,NaN
7,15Yx5Y/20Yx5Y,10.0,neutral,1.000,12,always,0.8,0.315,0.131,4105441.672,41.054,0.158,10.917,35.063,5.992,0.087,0.462,101.715,0.309,1.0,253.0,-0.458,0.529,-3287785.427,NaN
8,15Yx5Y/20Yx5Y,30.0,always_decrease,1.000,12,always,0.8,0.310,0.126,4164278.880,41.643,1.008,9.501,35.063,6.580,0.087,0.462,101.715,0.309,1.0,9.0,-0.436,0.523,-3658033.323,NaN
9,15Yx5Y/20Yx5Y,20.0,always_decrease,1.000,12,always,0.8,0.308,0.122,4145564.426,41.456,1.085,9.247,35.063,6.393,0.087,0.462,101.715,0.309,1.0,16.0,-0.418,0.526,-3616020.958,NaN



top 15 by direction-free Sharpe, carry unconstrained:


,pair,threshold_bp,resize_mode,beta,roll_months,entry_rule,be_ratio_max,sharpe,sharpe_ex_mtm,total_net_usd,net_bp,carry_bp,harvest_bp,mtm_bp,net_ex_mtm_bp,mean_carry_1y_bp,pct_days_carry_positive,mean_gamma_usd_bp2,mean_be_over_rv,occupancy,n_hedges,skew,hit_rate,max_dd_usd,citi_fig4_sharpe
0,10Yx10Y/20Yx10Y,20.0,always_decrease,1.025,24,be_ratio,0.5,0.098,0.243,736659.888,7.367,-3.588,62.758,-29.754,37.120,-3.552,0.024,204.012,0.751,0.227,0.0,0.843,0.499,-1465230.687,0.16
1,10Yx10Y/20Yx10Y,15.0,always_decrease,1.025,24,be_ratio,0.5,0.097,0.242,729311.579,7.293,-3.581,62.678,-29.754,37.047,-3.552,0.024,204.012,0.751,0.227,0.0,0.838,0.499,-1460130.293,0.16
2,10Yx10Y/20Yx10Y,25.0,always_decrease,1.025,24,be_ratio,0.5,0.097,0.242,726768.804,7.268,-3.595,62.667,-29.754,37.021,-3.552,0.024,204.012,0.751,0.227,0.0,0.849,0.499,-1460130.293,0.16
3,10Yx10Y/20Yx10Y,30.0,always_decrease,1.025,24,be_ratio,0.5,0.096,0.242,725631.994,7.256,-3.595,62.655,-29.754,37.010,-3.552,0.024,204.012,0.751,0.227,0.0,0.849,0.499,-1461267.104,0.16
4,10Yx10Y/20Yx10Y,10.0,always_decrease,1.025,24,be_ratio,0.5,0.093,0.240,702049.455,7.020,-3.583,62.407,-29.754,36.774,-3.552,0.024,204.012,0.751,0.227,0.0,0.830,0.499,-1463749.505,0.16
5,10Yx10Y/20Yx10Y,20.0,always_decrease,1.000,24,be_ratio,0.5,0.156,0.239,1248762.460,12.488,-3.514,61.228,-23.176,35.663,-3.552,0.024,204.012,0.751,0.227,0.0,0.865,0.490,-1529256.966,0.16
6,10Yx10Y/20Yx10Y,15.0,always_decrease,1.000,24,be_ratio,0.5,0.155,0.239,1241593.377,12.416,-3.508,61.149,-23.176,35.592,-3.552,0.024,204.012,0.751,0.227,0.0,0.863,0.487,-1529256.966,0.16
7,10Yx10Y/20Yx10Y,25.0,always_decrease,1.000,24,be_ratio,0.5,0.154,0.238,1239112.621,12.391,-3.521,61.138,-23.176,35.567,-3.552,0.024,204.012,0.751,0.227,0.0,0.870,0.490,-1529256.966,0.16
8,10Yx10Y/20Yx10Y,30.0,always_decrease,1.000,24,be_ratio,0.5,0.154,0.238,1238003.538,12.380,-3.521,61.127,-23.176,35.556,-3.552,0.024,204.012,0.751,0.227,0.0,0.870,0.490,-1529256.966,0.16
9,10Yx10Y/20Yx10Y,15.0,neutral,1.025,24,be_ratio,0.5,0.026,0.238,165629.466,1.656,-3.920,57.813,-29.754,31.410,-3.552,0.024,204.012,0.751,0.227,36.0,1.358,0.499,-1544384.238,0.16


## 7. Stability, not a winning cell

A single best cell out of several thousand is a lottery ticket. What matters is whether
the surface is flat where Citi says it is flat.

> "we have found that the Sharpe ratio of the strategy doesn't change
> significantly if the threshold is chosen in the **15-30bp range**, but
> declines with a smaller or larger threshold."

That is a falsifiable prediction about the shape of the threshold axis, made
on a different sample. Section 7.1 tests it directly.

In [13]:
STAB = (R1[(R1.entry_rule == "always") & (R1.resize_mode == "neutral")
           & (R1.beta == 1.0) & (R1.roll_months == 12)]
        .pivot(index="pair", columns="threshold_bp", values="sharpe"))
STAB["range"] = STAB.max(axis=1) - STAB.min(axis=1)
STAB["best_th"] = STAB[GRID["thresholds"]].idxmax(axis=1)
display(STAB.round(3).sort_values(15.0, ascending=False))

_mid = [t for t in GRID["thresholds"] if 15 <= t <= 30]
_out = [t for t in GRID["thresholds"] if t < 15 or t > 30]
_pred = pd.DataFrame({
    "sharpe_15_30bp": STAB[_mid].mean(axis=1),
    "sharpe_10_and_40bp": STAB[_out].mean(axis=1),
    "spread_inside_band": STAB[_mid].max(axis=1) - STAB[_mid].min(axis=1),
})
_pred["citi_claim_holds"] = _pred.sharpe_15_30bp >= _pred.sharpe_10_and_40bp
display(_pred.round(3))
_held, _n = int(_pred.citi_claim_holds.sum()), len(_pred)
print(f"\nCiti's 15-30bp plateau claim holds on {_held}/{_n} pairs; "
      f"mean Sharpe inside the band {_pred.sharpe_15_30bp.mean():+.3f} "
      f"vs {_pred.sharpe_10_and_40bp.mean():+.3f} outside it. "
      f"Median spread WITHIN the band: {_pred.spread_inside_band.median():.3f}.")

threshold_bp,10.0,15.0,20.0,25.0,30.0,40.0,range,best_th
pair,,,,,,,,
15Yx10Y/25Yx10Y,0.611,0.623,0.597,0.578,0.627,0.633,0.055,40.0
15Yx5Y/25Yx10Y,0.579,0.590,0.564,0.546,0.595,0.602,0.056,40.0
20Yx5Y/25Yx10Y,0.562,0.574,0.554,0.536,0.578,0.583,0.047,40.0
15Yx5Y/20Yx15Y,0.535,0.537,0.524,0.505,0.527,0.560,0.055,40.0
15Yx10Y/25Yx5Y,0.500,0.490,0.486,0.470,0.519,0.464,0.055,30.0
20Yx5Y/25Yx5Y,0.494,0.487,0.481,0.464,0.514,0.460,0.054,30.0
10Yx10Y/25Yx10Y,0.449,0.464,0.430,0.404,0.474,0.480,0.076,40.0
15Yx5Y/25Yx5Y,0.474,0.464,0.460,0.447,0.491,0.441,0.050,30.0
15Yx5Y/20Yx10Y,0.430,0.440,0.411,0.404,0.434,0.395,0.045,15.0


,sharpe_15_30bp,sharpe_10_and_40bp,spread_inside_band,citi_claim_holds
pair,,,,
10Yx10Y/15Yx15Y,0.128,0.126,0.031,True
10Yx10Y/20Yx10Y,0.255,0.241,0.044,True
10Yx10Y/20Yx15Y,0.371,0.404,0.041,False
10Yx10Y/20Yx5Y,0.111,0.106,0.068,True
10Yx10Y/25Yx10Y,0.443,0.464,0.070,False
10Yx10Y/25Yx5Y,0.319,0.309,0.055,True
15Yx10Y/25Yx10Y,0.606,0.622,0.050,False
15Yx10Y/25Yx5Y,0.491,0.482,0.048,True
15Yx5Y/20Yx10Y,0.422,0.413,0.036,True



Citi's 15-30bp plateau claim holds on 9/15 pairs; mean Sharpe inside the band +0.404 vs +0.408 outside it. Median spread WITHIN the band: 0.044.


In [14]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

_hm = STAB[GRID["thresholds"]].sort_index()
fig = go.Figure(go.Heatmap(z=_hm.values, x=[f"{t:g}bp" for t in _hm.columns], y=_hm.index,
                           colorscale="RdYlGn", zmid=0,
                           text=np.round(_hm.values, 2), texttemplate="%{text}",
                           colorbar=dict(title="Sharpe")))
fig.update_layout(template="plotly_dark", height=620,
                  title="annualised Sharpe by pair x hedge threshold "
                        "(always-on, DV01-neutral, annual roll, net of Citi Fig-9 costs)")
fig.show()

### 7.1 The other axes

Each knob's marginal effect, averaged over everything else. A knob whose
marginal effect is inside the noise is a knob that should be left alone.

In [15]:
for axis in ("resize_mode", "beta", "roll_months", "entry_rule", "cost_multiplier"):
    src = RESULTS if axis == "cost_multiplier" else R1
    t = src.groupby(axis).agg(n=("sharpe", "size"), mean_sharpe=("sharpe", "mean"),
                              median_sharpe=("sharpe", "median"),
                              mean_sharpe_ex_mtm=("sharpe_ex_mtm", "mean"),
                              mean_net_usd=("total_net_usd", "mean"),
                              mean_carry_bp=("carry_bp", "mean"),
                              mean_harvest_bp=("harvest_bp", "mean"),
                              mean_net_ex_mtm_bp=("net_ex_mtm_bp", "mean"),
                              mean_occupancy=("occupancy", "mean"))
    print(f"\n--- {axis} ---")
    display(t.round(3))


--- resize_mode ---


,n,mean_sharpe,median_sharpe,mean_sharpe_ex_mtm,mean_net_usd,mean_carry_bp,mean_harvest_bp,mean_net_ex_mtm_bp,mean_occupancy
resize_mode,,,,,,,,,
always_decrease,2520,-0.047,0.002,-0.552,846144.373,-3.552,10.168,-37.922,0.429
neutral,2520,-0.055,-0.021,-0.584,683398.932,-4.497,9.931,-39.549,0.429



--- beta ---


,n,mean_sharpe,median_sharpe,mean_sharpe_ex_mtm,mean_net_usd,mean_carry_bp,mean_harvest_bp,mean_net_ex_mtm_bp,mean_occupancy
beta,,,,,,,,,
1.000,2520,-0.043,-0.010,-0.573,830294.834,-3.866,9.925,-38.697,0.429
1.025,2520,-0.060,-0.015,-0.563,699248.470,-4.183,10.173,-38.775,0.429



--- roll_months ---


,n,mean_sharpe,median_sharpe,mean_sharpe_ex_mtm,mean_net_usd,mean_carry_bp,mean_harvest_bp,mean_net_ex_mtm_bp,mean_occupancy
roll_months,,,,,,,,,
12,2520,-0.029,0.021,-0.660,995009.409,-3.353,11.511,-36.940,0.429
24,2520,-0.073,-0.038,-0.476,534533.896,-4.696,8.588,-40.531,0.429



--- entry_rule ---


,n,mean_sharpe,median_sharpe,mean_sharpe_ex_mtm,mean_net_usd,mean_carry_bp,mean_harvest_bp,mean_net_ex_mtm_bp,mean_occupancy
entry_rule,,,,,,,,,
always,720,0.378,0.413,-0.048,5545868.835,-14.506,10.353,-7.726,1.000
be_ratio,1440,0.181,0.322,-0.398,2663516.849,-2.065,18.455,-34.816,0.633
carry,720,-0.259,-0.304,-1.166,-1490050.920,2.291,1.085,-51.259,0.237
z,1440,-0.293,-0.258,-0.742,-1472998.850,-5.304,7.206,-50.610,0.184
z_and_be,720,-0.252,-0.140,-0.483,-1083452.344,-1.219,7.584,-41.311,0.132



--- cost_multiplier ---


,n,mean_sharpe,median_sharpe,mean_sharpe_ex_mtm,mean_net_usd,mean_carry_bp,mean_harvest_bp,mean_net_ex_mtm_bp,mean_occupancy
cost_multiplier,,,,,,,,,
0.0,5040,0.620,0.644,0.084,5240804.759,-4.025,10.049,6.025,0.429
1.0,5040,-0.051,-0.012,-0.568,764771.652,-4.025,10.049,-38.736,0.429
2.0,5040,-0.585,-0.568,-0.977,-3711261.454,-4.025,10.049,-83.496,0.429


### 7.2 Carry against Sharpe — the efficient frontier the brief asks for

One point per pair at Citi's own specification. Up-and-to-the-right is
"long vol and earns theta"; the bottom-left quadrant is long vol that pays
handsomely for the privilege.

In [16]:
_pts = BASE.reset_index()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=_pts.mean_carry_1y_bp, y=_pts.sharpe, mode="markers+text",
    text=_pts.pair, textposition="top center", textfont=dict(size=9),
    marker=dict(size=np.clip(_pts.mean_gamma_usd_bp2 / 12, 6, 30),
                color=_pts.pct_days_carry_positive, colorscale="Viridis",
                showscale=True, colorbar=dict(title="% days<br>carry >= 0")),
    hovertemplate="%{text}<br>carry %{x:.2f}bp/yr<br>Sharpe %{y:.3f}<extra></extra>"))
fig.add_vline(x=0, line=dict(color="#ff5c5c", dash="dash"))
fig.add_hline(y=0, line=dict(color="#7b8394", dash="dot"))
fig.update_layout(template="plotly_dark", height=620,
                  xaxis_title="mean ex-ante 1y carry (bp) -- right of the red line is PAID to hold",
                  yaxis_title="annualised Sharpe, net of Citi Fig-9 costs",
                  title="marker size = mean $ gamma per bp^2; every pair here is long gamma")
fig.show()

## 8. Equity curves of the cells worth looking at

In [17]:
from BT.trade_dashboard import compare_curves


def _book(pair, rm, th, beta, mode, entry=None, mult=PRIMARY_COST):
    led = LEDGERS[tuple(pair.split("/"))][(rm, th, beta, mode)]
    led = led[(led.index >= pd.Timestamp(GRID["start"])) & (led.index <= pd.Timestamp(GRID["end"]))]
    cfg = S3.Strat3Config(short_leg=pair.split("/")[0], long_leg=pair.split("/")[1],
                          roll_months=rm, hedge_threshold_bp=th, beta=beta,
                          resize_mode=mode, **(entry or {"entry_rule": "always"}))
    state = None if cfg.entry_rule == "always" else S3.entry_state(SCREENS[pair], cfg, led.index)
    st = S3.book_stats(led, state, costs=S3.cost_schedule_for(*cfg.pair, multiplier=mult),
                       package_dv01_usd=GRID["package_dv01_usd"])
    eq = st["equity"]
    d = eq.diff().fillna(eq.iloc[0])
    return pd.DataFrame({"timestamp": eq.index, "pnl": d.to_numpy()}), st


_curves, _rows = {}, []
for pair in LIKE.index[:6]:
    b, st = _book(pair, 12, 25.0, 1.0, "neutral")
    _curves[pair] = b
    _rows.append({"pair": pair, **{k: st[k] for k in
                                   ("total_net_usd", "sharpe", "carry_bp", "harvest_bp",
                                    "mtm_bp", "n_hedges", "max_dd_usd")}})
fig = compare_curves(_curves, title="top six pairs at Citi's own 25bp specification, "
                                    "net of Fig-9 costs")
fig.show()
display(pd.DataFrame(_rows).set_index("pair").round(3))

,total_net_usd,sharpe,carry_bp,harvest_bp,mtm_bp,n_hedges,max_dd_usd
pair,,,,,,,
15Yx10Y/25Yx10Y,7869444.054,0.578,-6.181,11.320,78.205,38.0,-2496182.040
15Yx5Y/25Yx10Y,9788897.859,0.546,-6.238,13.745,95.234,38.0,-3977528.284
20Yx5Y/25Yx10Y,5827754.598,0.536,-6.259,8.789,60.171,38.0,-1401372.544
15Yx5Y/20Yx15Y,7779971.937,0.505,-3.626,13.114,73.053,46.0,-3723288.016
15Yx10Y/25Yx5Y,5937357.174,0.470,-2.592,10.810,55.656,42.0,-2572699.634
20Yx5Y/25Yx5Y,3793409.179,0.464,-2.662,7.221,37.622,42.0,-1388298.389


In [18]:
_pair = LIKE.index[0]
_curves2 = {f"@{t:g}bp": _book(_pair, 12, t, 1.0, "neutral")[0] for t in GRID["thresholds"]}
_curves2["always_decrease @25bp"] = _book(_pair, 12, 25.0, 1.0, "always_decrease")[0]
_curves2["beta 1.025 @25bp"] = _book(_pair, 12, 25.0, 1.025, "neutral")[0]
fig = compare_curves(_curves2, title=f"{_pair}: the hedge threshold and the resize mode")
fig.show()

## 9. What the search cost

The best Sharpe out of several thousand tries is not the Sharpe to expect
from it out of sample. The deflated Sharpe ratio (Bailey & López de Prado)
discounts the maximum by the expected maximum of that many draws under a null
of zero skill, using the winning cell's OWN daily return moments — not the
cross-section's — for the non-normality correction.

In [19]:
from scipy import stats as _st

_s = R1.sharpe.dropna()
_n_trials = len(R1)
_win = R1.loc[R1.sharpe.idxmax()]
_win_book, _win_stats = _book(_win.pair, int(_win.roll_months), float(_win.threshold_bp),
                              float(_win.beta), _win.resize_mode,
                              entry={"entry_rule": _win.entry_rule,
                                     "z_window": _win.z_window, "z_min": float(_win.z_min),
                                     "be_ratio_max": float(_win.be_ratio_max)})
_r = _win_stats["net_daily"]
_T = len(_r)
_sr = float(_r.mean() / _r.std(ddof=1))          # per-day Sharpe, the DSR's unit
_g3, _g4 = float(_r.skew()), float(_r.kurt() + 3.0)
_sr_std_cs = float(_s.std(ddof=1) / np.sqrt(252.0))   # cross-section, per-day units
_e_max = _sr_std_cs * ((1 - np.euler_gamma) * _st.norm.ppf(1 - 1.0 / _n_trials)
                       + np.euler_gamma * _st.norm.ppf(1 - 1.0 / (_n_trials * np.e)))
_z = ((_sr - _e_max) * np.sqrt(_T - 1)
      / np.sqrt(1 - _g3 * _sr + (_g4 - 1) / 4.0 * _sr ** 2))
display(pd.Series({
    "cells searched (at 1x cost)": _n_trials,
    "winning cell": f"{_win.pair} @{_win.threshold_bp:g}bp {_win.resize_mode} "
                    f"beta={_win.beta} roll={_win.roll_months}m {_win.entry_rule}",
    "sample days": _T,
    "winner annualised Sharpe": _sr * np.sqrt(252.0),
    "winner daily skew": _g3,
    "winner daily kurtosis": _g4,
    "cross-sectional Sharpe sd (annualised)": float(_s.std(ddof=1)),
    "expected max Sharpe under the null (annualised)": _e_max * np.sqrt(252.0),
    "deflated Sharpe p(true SR > 0)": float(_st.norm.cdf(_z)),
    "median cell Sharpe": float(_s.median()),
    "% of cells with Sharpe > 0": float(100 * (_s > 0).mean()),
    "Citi Fig-4 mean Sharpe (their sample)": float(np.mean(list(S3.CITI_FIG4_SHARPE.values()))),
}).to_frame("value"))

,value
cells searched (at 1x cost),5040
winning cell,10Yx10Y/25Yx10Y @40bp neutral beta=1.025 roll=...
sample days,1898
winner annualised Sharpe,0.765305
winner daily skew,0.263963
winner daily kurtosis,12.849028
cross-sectional Sharpe sd (annualised),0.433308
expected max Sharpe under the null (annualised),1.59879
deflated Sharpe p(true SR > 0),0.010909
median cell Sharpe,-0.012078


## 10. Reading this notebook

**Every cell in this grid is long gamma.** Section 4 checks it 50,955 times.
So "is it long vol" is not the discriminating question and no ranking here
turns on it. The discriminating question is what the position pays to be
long gamma, which is the carry column, and that is set by which forward
points the pair straddles — not by any hedging knob.

**The gated entry rules scale an aged ledger.** A cell with `entry_rule != "always"`
multiplies the always-on ledger's daily flows by a lag-1 {0,1} state and
charges an initiation on every flip. It does not strike a fresh package at
each entry. That understates the cost of re-entering at market and overstates
the age of the position at entry; it is the same approximation
`scripts/sv_citivelo_h13.py` documents. Treat a gated cell's Sharpe as an
upper bound on the gated idea, and treat the always-on rows as the numbers
that carry no such caveat.

**Citi's Figure-4 Sharpes are on a different sample.** 12/2013-5/2019 versus
our 1/2019-8/2026. The comparison in section 6.3 is a robustness read across
two regimes, not a reproduction; the sample here contains the 2020 collapse
and the 2022-2023 selloff, both of which move a convexity book far more than
anything in theirs.

**Costs are first-order.** Section 7.1's `cost_multiplier` row is not a
sensitivity, it is a result. Anything that survives only at multiplier 0 is
an artifact of not paying to trade.